# Timestep Sensitivity — 60 min vs. 15 min

**Purpose:** Tests whether the simulation timestep (60 min vs. 15 min)
significantly affects the peak flood hydrograph, to justify the chosen
temporal resolution.

**What it does:**
- Loads TALSIM results at 60-minute and 15-minute timesteps
- Compares results across three IC scenarios (dry / medium / saturated)
- Produces a 4-panel figure:
  - (a) 60 min | T = 500yr     (b) 15 min | T = 500yr
  - (c) 60 min | T = 5000yr    (d) 15 min | T = 5000yr
- Color ramp ranked by peak discharge

**User settings:** Edit only the USER SETTINGS block at the top  
**Input:** TALSIM `.WEL` output folders for both timesteps  
**Output:** 4-panel timestep comparison PNG

---

In [ ]:
IC_SCENARIOS = [
    ("Initial_saturation = 21.58 % ",       Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_21.58_60min")),
    ("Initial_saturation = 41.71 % ",    Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_41.71_60min\DVWK")),
    ("Initial_saturation = 100 %   ", Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_100_60min")),
]


In [ ]:
# =============================================================================
# FIGURE — Timestep Comparison: 60 min vs 15 min
# 4-panel figure:
#   (a) 60 min | T = 500yr      (b) 15 min | T = 500yr
#   (c) 60 min | T = 5000yr     (d) 15 min | T = 5000yr
#
# Each panel shows all selected storm durations.
# Color ramp: peak-ranked (highest peak = red).
#
# HOW TO USE:
#   Edit ONLY the USER SETTINGS block below, then run.
# =============================================================================

# %% Imports
from pathlib import Path
import re
import unicodedata
import datetime as dt

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator

# =============================================================================
# *** USER SETTINGS — EDIT ONLY HERE ***
# =============================================================================

# --- Folders ----------------------------------------------------------------
FOLDER_60MIN = Path(r"C:\Users\raah\Desktop\Project_ZR\Ziegenrück_nr\Ziegenrück_100_60min")
FOLDER_15MIN = Path(r"C:\Users\raah\Desktop\Project_ZR\Ziegenrück_nr\Ziegenrück_100_15min")

# --- WEL file name inside each event subfolder ------------------------------
WEL_60MIN = "Ziegenrück_nr.WEL"
WEL_15MIN = "Ziegenrück_nr.WEL"

# --- Column to extract ------------------------------------------------------
COL = "S020_1ZU"

# --- Return periods ---------------------------------------------------------
T1 = 500
T2 = 5000

# --- Storm durations to include [hours] -------------------------------------
DURATIONS_TO_PLOT = [1, 2, 3, 6, 12, 18, 24, 48, 72]

# --- Line widths ------------------------------------------------------------
LW_CRITICAL = 1
LW_NORMAL   = 0.9

# --- Output folder ----------------------------------------------------------
OUT_DIR = Path(r"C:\Users\raah\Desktop\Project_ZR\figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Fixed settings
# =============================================================================

re_folder = re.compile(
    r"^(?P<num>\d{3})_(?P<dauer>\d+(?:[.,]\d+)?)h_(?P<yr>\d+)yr$",
    re.IGNORECASE
)

# =============================================================================
# %% Helpers
# =============================================================================

def ascii_safe(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")

def merge_split_underscore_cols(cols):
    fixed, i = [], 0
    while i < len(cols):
        if (i + 1 < len(cols)
                and cols[i + 1].startswith("_")
                and re.match(r"^[A-Za-z0-9]+$", cols[i])):
            fixed.append(cols[i] + cols[i + 1]); i += 2
        else:
            fixed.append(cols[i]); i += 1
    return fixed

_date_re = re.compile(r"^\d{2}\.\d{2}\.\d{4}$")
_time_re = re.compile(r"^\d{2}:\d{2}$")

def _find_wel(folder, wel_name):
    for name in [wel_name, wel_name.lower(), wel_name + ".txt"]:
        p = folder / name
        if p.exists():
            return p
    raise FileNotFoundError(f"WEL not found in {folder}")

def parse_wel(path, value_col):
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    header_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("Datum_Zeit"):
            header_idx = i; break
    if header_idx is None:
        raise ValueError(f"Header not found in {path}")
    cols = merge_split_underscore_cols(lines[header_idx].split())
    if value_col not in cols:
        raise KeyError(
            f"Column '{value_col}' not in {path.name}\n"
            f"Available: {[c for c in cols if '1ZU' in c or '1AB' in c]}"
        )
    vpos            = cols.index(value_col)
    expected_tokens = len(cols) + 1
    start = header_idx + 1
    while start < len(lines):
        s = lines[start].strip()
        if not s or s.startswith("*") or s.lstrip().startswith("-"):
            start += 1; continue
        if lines[start].split() and _date_re.match(lines[start].split()[0]):
            break
        start += 1

    def find_dt(buf, at=0):
        for j in range(at, len(buf) - 1):
            if _date_re.match(buf[j]) and _time_re.match(buf[j + 1]):
                return j
        return None

    times, values, buf = [], [], []
    for line in lines[start:]:
        if not line.strip() or line.strip().startswith("*"):
            continue
        buf.extend(line.split())
        while True:
            s0 = find_dt(buf)
            if s0 is None:
                if len(buf) > 10 * expected_tokens: buf = buf[-expected_tokens:]
                break
            if s0 > 0: buf = buf[s0:]
            s1 = find_dt(buf, 2)
            end = s1 if (s1 and s1 < expected_tokens) else expected_tokens
            if len(buf) < end: break
            rec = buf[:end]; buf = buf[end:]
            if len(rec) != expected_tokens: continue
            try:
                ts = dt.datetime.strptime(rec[0] + " " + rec[1], "%d.%m.%Y %H:%M")
            except Exception:
                ts = None
            try:
                val = float(rec[2:][vpos - 1].replace(",", "."))
            except Exception:
                continue
            times.append(ts); values.append(val)

    q = np.asarray(values, dtype=float)
    if times and all(t is not None for t in times):
        t0  = times[0]
        t_h = np.array([(t - t0).total_seconds() / 3600 for t in times])
    else:
        t_h = np.arange(len(q), dtype=float)
    return {"q": q, "t_h": t_h}


def load_selected_durations(root, wel_name, value_col, return_period, durations):
    """Load selected storm durations for a given return period."""
    results = {}
    for sf in sorted(root.iterdir()):
        if not sf.is_dir(): continue
        m = re_folder.match(sf.name)
        if not m: continue
        if int(m.group("yr")) != return_period: continue
        dauer_h = float(m.group("dauer").replace(",", "."))
        if not any(abs(dauer_h - d) < 0.01 for d in durations):
            continue
        try:
            wel    = _find_wel(sf, wel_name)
            parsed = parse_wel(wel, value_col)
            if len(parsed["q"]) > 0:
                parsed["qmax"] = float(np.nanmax(parsed["q"]))
                results[dauer_h] = parsed
        except Exception as e:
            print(f"  [SKIP] {sf.name}: {e}")
    if not results:
        print(f"  [WARNING] No data for T={return_period}yr in {root.name}")
    return results


# =============================================================================
# %% Color ramp — peak-ranked
# =============================================================================

def make_colors(dauer_list, data):
    if not dauer_list:
        return {}, None
    critical = max(dauer_list, key=lambda d: data[d]["qmax"])
    non_crit = [d for d in dauer_list if d != critical]
    nc       = len(non_crit)
    RAMP = mcolors.LinearSegmentedColormap.from_list("hydro_ramp", [
        "#08306b", "#2196c8", "#1a6b2f",
        "#78c44a", "#f5e400", "#f57c00", "#7b3a10",
    ])
    colors = {d: RAMP(i / max(nc - 1, 1)) for i, d in enumerate(non_crit)}
    colors[critical] = "#e00000"
    return colors, critical


# =============================================================================
# %% Panel plotter
# =============================================================================

def plot_panel(ax, data, panel_title):
    """Plot all selected durations on one panel."""
    dauer_list = sorted(data.keys(), key=float)
    if not dauer_list:
        ax.set_title(f"{panel_title}\n(no data)", pad=7)
        return

    colors, critical = make_colors(dauer_list, data)
    qmax_global = max(data[d]["qmax"] for d in dauer_list)

    for d in dauer_list:
        q       = data[d]["q"]
        t       = data[d]["t_h"]
        is_crit = (d == critical)
        ax.plot(
            t, q,
            color     = colors[d],
            linestyle = "-",
            linewidth = LW_CRITICAL if is_crit else LW_NORMAL,
            alpha     = 1.0 if is_crit else 0.82,
            label     = f"{d:g} h ★" if is_crit else f"{d:g} h",
            zorder    = 50 if is_crit else 2,
        )
        # Peak triangle
        idx = int(np.nanargmax(q))
        ax.plot(t[idx], q[idx],
                marker="^",
                markersize = 5 if is_crit else 3.5,
                color      = colors[d],
                markeredgewidth = 0.5,
                markeredgecolor = "white",
                zorder     = 60 if is_crit else 5)

    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0, top=qmax_global * 1.15)
    ax.xaxis.set_major_locator(MaxNLocator(8, integer=False))
    ax.tick_params(axis="both", length=4)
    ax.set_xlabel("Time [h] (from event start)", fontsize=8)
    ax.set_ylabel(f"Discharge [m³/s]", fontsize=8)
    ax.set_title(panel_title, pad=7, fontsize=9, loc="left")

    leg = ax.legend(
        title="Storm duration",
        ncol=2, loc="upper right",
        framealpha=0.92, edgecolor="0.7",
        borderpad=0.5, labelspacing=0.25,
        handlelength=1.6, fontsize=7,
    )
    leg.get_title().set_fontsize(7)
    leg.get_title().set_fontstyle("italic")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# =============================================================================
# %% Main
# =============================================================================

plt.rcParams.update({
    "font.family":    "DejaVu Sans",
    "font.size":       8,
    "axes.titlesize":  9,
    "axes.labelsize":  8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.5,
    "axes.linewidth":  0.7,
    "grid.linewidth":  0.45,
    "grid.alpha":      0.35,
    "grid.linestyle":  "--",
    "axes.grid":       True,
    "axes.axisbelow":  True,
    "figure.dpi":      150,
})

print("=" * 60)
print("Timestep Comparison: 60 min vs 15 min")
print(f"Column: {COL}")
print("=" * 60)

print(f"\nLoading 60 min | T = {T1} yr ...")
d_60_T1 = load_selected_durations(FOLDER_60MIN, WEL_60MIN, COL, T1, DURATIONS_TO_PLOT)

print(f"Loading 15 min | T = {T1} yr ...")
d_15_T1 = load_selected_durations(FOLDER_15MIN, WEL_15MIN, COL, T1, DURATIONS_TO_PLOT)

print(f"Loading 60 min | T = {T2} yr ...")
d_60_T2 = load_selected_durations(FOLDER_60MIN, WEL_60MIN, COL, T2, DURATIONS_TO_PLOT)

print(f"Loading 15 min | T = {T2} yr ...")
d_15_T2 = load_selected_durations(FOLDER_15MIN, WEL_15MIN, COL, T2, DURATIONS_TO_PLOT)

# -----------------------------------------------------------------------
# Layout:
#   (a) 60 min | T = T1      (b) 15 min | T = T1
#   (c) 60 min | T = T2      (d) 15 min | T = T2
# -----------------------------------------------------------------------
FIG_WIDTH  = 6.30
FIG_HEIGHT = 5.50
fig, axes  = plt.subplots(2, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))

print("\nPlotting panels...")

plot_panel(axes[0, 0], d_60_T1, f"(a)  60 min  |  T = {T1} yr")
plot_panel(axes[0, 1], d_15_T1, f"(b)  15 min  |  T = {T1} yr")
plot_panel(axes[1, 0], d_60_T2, f"(c)  60 min  |  T = {T2} yr")
plot_panel(axes[1, 1], d_15_T2, f"(d)  15 min  |  T = {T2} yr")

fig.tight_layout()
fig.subplots_adjust(hspace=0.42, wspace=0.32)

outpng = OUT_DIR / f"Fig_Timestep_Comparison_{COL}.png"
fig.savefig(outpng, dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"\nSaved: {outpng}")
print("In Word: Insert > Pictures > set width = 16.0 cm")

# =============================================================================
# %% COLUMN HELPER — uncomment to check available columns
# =============================================================================
#
# for sf in sorted(FOLDER_60MIN.iterdir()):
#     if sf.is_dir():
#         wel = sf / WEL_60MIN
#         if wel.exists():
#             lines = wel.read_text(encoding="utf-8", errors="replace").splitlines()
#             for line in lines:
#                 if line.strip().startswith("Datum_Zeit"):
#                     cols = line.split()
#                     print("1ZU (inflow) :", [c for c in cols if "1ZU" in c])
#                     print("1AB (outflow):", [c for c in cols if "1AB" in c])
#                     break
#         break